In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
laptimes_df=spark.read\
    .option("inferSchema", True)\
        .csv("/Volumes/formula1_dev/bronze/demo_volume/source_files/lap_times/")
laptimes_df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
lap_times_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                      StructField("driverId", IntegerType(), True),
                                      StructField("lap", IntegerType(), True),
                                      StructField("position", IntegerType(), True),
                                      StructField("time", StringType(), True),
                                      StructField("milliseconds", IntegerType(), True)
                                     ])

In [0]:
laptimes_df=spark.read\
    .schema(lap_times_schema)\
        .csv("/Volumes/formula1_dev/bronze/demo_volume/source_files/lap_times/")
laptimes_df.display()

In [0]:
from pyspark.sql.functions import lit, current_timestamp,current_date
laptimes_df1= laptimes_df.withColumnRenamed("driverId", "driver_id")\
    .withColumnRenamed("raceId", "race_id")\
    .withColumn("ingestion_timestamp", current_timestamp())\
        .withColumn("ingestion_date", current_date())

In [0]:
laptimes_df1.display()

In [0]:
# laptimes_df1.write.mode("overwrite").format("delta").option("")
laptimes_df1.write.mode("overwrite").format("delta")\
    .option("path", "abfss://raw@formula1adls.dfs.core.windows.net/lap_times").saveAsTable(f"formula1_{env}.bronze.lap_times")


In [0]:
%sql
select * from formula1_dev.bronze.lap_times